# 09 · Sentimento por segmento: algum tipo de texto engana?

Nas etapas 04 e 05, o sentimento entrou como uma média geral. Aqui ele é separado por **tipo de texto**, para ver se algum segmento ajuda e se algum atrapalha:
- **Veículo da notícia:** CoinTelegraph, CryptoNews e CryptoPotato.
- **Assunto da notícia:** bitcoin, altcoin, blockchain, ethereum, nft e defi.
- **Força:** textos fortes (|nota| ≥ 0,5) e fracos.
- **Engajamento do tweet:** populares (25% mais curtidos no treino) e comuns.

Para cada segmento e moeda, o notebook faz dois testes:
1. **Sinal:** o sentimento do segmento acompanha o retorno de amanhã? Um segmento com correlação **negativa** aponta para o lado errado: é o candidato a "enganoso".
2. **Modelo:** a LSTM e o Transformer (hiperparâmetros das etapas 02 e 03, 5 sementes) melhoram ou pioram com esse segmento, contra a versão financeira da etapa 05.

Como são muitos testes ao mesmo tempo, os p são ajustados por Benjamini-Hochberg: sem isso, uns 5% dariam "significativo" por acaso.

In [ ]:
import os, sys, glob
achados = glob.glob('../src/tcc/config.py') + glob.glob('/kaggle/input/**/src/tcc/config.py', recursive=True)
RAIZ = achados[0].rsplit('/src/', 1)[0]              # pasta do projeto (no PC ou no dataset do Kaggle)
sys.path.insert(0, os.path.join(RAIZ, 'src'))
import warnings; warnings.filterwarnings('ignore')
import json, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from tcc import avaliacao, config, dados, experimento, segmentos, sentimento, textos

In [ ]:
ALVO   = 'retorno'
INICIO = '2022-01-01'
FIM    = '2023-12-31'
MOEDAS = config.MOEDAS
MIN_DIAS = 60          # dias com texto para testar o sinal
MIN_DIAS_MODELO = 100  # dias com texto para testar no modelo

pasta = lambda etapa: os.path.join(RAIZ, 'results', etapa)
MODELOS = {'LSTM': json.load(open(os.path.join(pasta('02_lstm'), 'hiperparametros.json'))),
           'Transformer': json.load(open(os.path.join(pasta('03_transformer'), 'hiperparametros.json')))}
ETAPA_05 = pd.read_csv(os.path.join(pasta('05_com_sentimento'), 'previsoes.csv'), parse_dates=['data'])
NOTAS = pd.read_csv(os.path.join(pasta('04_sentimento'), 'nota_por_texto.csv'), parse_dates=['data'])

SAIDA = config.PASTA_RESULTADOS / '09_segmentos'
SAIDA.mkdir(parents=True, exist_ok=True)

## 1. Textos com veículo, assunto e curtidas

As notas do CryptoBERT vêm da etapa 04. Só um texto que ainda não tenha nota é classificado de novo.

In [ ]:
longa = textos.por_moeda(INICIO, FIM).merge(NOTAS[['moeda', 'fonte', 'data', 'texto', 'nota']], how='left')
faltam = longa.loc[longa['nota'].isna(), 'texto'].drop_duplicates().tolist()
print(f'{len(longa) - longa.nota.isna().sum()} de {len(longa)} textos com nota da etapa 04; {len(faltam)} para classificar')
if faltam:
    longa.loc[longa['nota'].isna(), 'nota'] = longa.loc[longa['nota'].isna(), 'texto'].map(dict(zip(faltam, sentimento.classificar(faltam))))

dias = pd.date_range(INICIO, FIM, freq='D')
fim_treino = dias[experimento.cortes(len(dias) - 1)[0]]
mascaras = segmentos.marcar(longa, fim_treino)
print('Fim do treino:', fim_treino.date(), '| segmentos:', list(mascaras))

contagem = pd.DataFrame([{'segmento': s, 'moeda': m, 'textos': int((mascaras[s] & (longa.moeda == m)).sum()),
                          'dias_com_texto': longa.loc[mascaras[s] & (longa.moeda == m), 'data'].nunique(),
                          'nota_media': longa.loc[mascaras[s] & (longa.moeda == m), 'nota'].mean()}
                         for s in mascaras for m in MOEDAS])
contagem.round(3).to_csv(SAIDA / 'contagem_segmentos.csv', index=False)
contagem.pivot(index='segmento', columns='moeda', values='dias_com_texto')[MOEDAS]

## 2. Sinal: o segmento acompanha o retorno de amanhã?

Spearman nos dias com texto, de 2022 a 2023. `acerto_%`: dias em que o sentimento acima da média do treino veio antes de alta (e abaixo, antes de queda). Abaixo de 50%, o segmento aponta mais para o lado errado.

In [ ]:
retornos = {m: dados.entradas(dados.precos(m, INICIO, FIM), ALVO)[1] for m in MOEDAS}
brutos, linhas = {}, []
for s in mascaras:
    for m in MOEDAS:
        b = brutos[s, m] = segmentos.diario(longa, mascaras[s], m, INICIO, FIM)
        r = retornos[m]
        ok = b.notna() & r.reindex(b.index).notna() & r.shift(-1).reindex(b.index).notna()
        if ok.sum() < MIN_DIAS:
            continue
        hoje, amanha = r.reindex(b.index)[ok], r.shift(-1).reindex(b.index)[ok]
        alto = b[ok] > b[:fim_treino].mean()
        linhas.append({'segmento': s, 'moeda': m, 'dias': int(ok.sum()),
                       'rho_hoje': spearmanr(b[ok], hoje)[0], 'rho_amanha': spearmanr(b[ok], amanha)[0],
                       'p_amanha': spearmanr(b[ok], amanha)[1], 'acerto_%': float(((amanha > 0) == alto).mean() * 100)})
sinal = pd.DataFrame(linhas)
sinal['p_amanha_ajustado'] = avaliacao.benjamini_hochberg(sinal['p_amanha'])
sinal['leitura'] = np.select([(sinal.p_amanha_ajustado < .05) & (sinal.rho_amanha < 0),
                              (sinal.p_amanha_ajustado < .05) & (sinal.rho_amanha > 0)],
                             ['aponta o lado errado', 'antecipa'], 'sem relação com amanhã')
sinal.round(4).to_csv(SAIDA / 'sinal_por_segmento.csv', index=False)
print(sinal['leitura'].value_counts())
sinal.sort_values('p_amanha').head(10).round(3)

## 3. Modelo: o segmento melhora ou piora a previsão?

In [ ]:
fin = ETAPA_05[ETAPA_05.variante == 'financeiro']
testes = [(s, m) for s in mascaras for m in MOEDAS if brutos[s, m].notna().sum() >= MIN_DIAS_MODELO]
print(len(testes), 'pares segmento × moeda com dados suficientes')
partes = []
for k, (s, m) in enumerate(testes, 1):
    extras = segmentos.preencher(brutos[s, m])
    for nome, hp in MODELOS.items():
        partes.append(experimento.rodar(m, ALVO, INICIO, FIM, hp=hp, extras=extras).assign(modelo=nome, variante=s))
    print(f'{k}/{len(testes)}', s, m, 'ok')
    if k % 5 == 0:
        pd.concat(partes, ignore_index=True).to_csv(SAIDA / 'previsoes.csv', index=False)   # guarda o progresso
prev = pd.concat([fin[fin.moeda.isin([m for _, m in testes])], *partes], ignore_index=True)
prev.to_csv(SAIDA / 'previsoes.csv', index=False)

In [ ]:
resumo = experimento.resumir(prev, chave=['modelo', 'variante'])
base = resumo[resumo.variante == 'financeiro'].set_index(['moeda', 'modelo'])['MAPE_%']
media = prev.groupby(['moeda', 'modelo', 'variante', 'data'])[['real', 'previsto']].mean()
linhas = []
for s, m in testes:
    for nome in MODELOS:
        f, g = media.loc[(m, nome, 'financeiro')], media.loc[(m, nome, s)]
        dm, p = avaliacao.diebold_mariano(g.previsto - g.real, f.previsto - f.real)
        mape = resumo.set_index(['moeda', 'modelo', 'variante']).loc[(m, nome, s), 'MAPE_%']
        linhas.append({'segmento': s, 'moeda': m, 'modelo': nome, 'MAPE_%': mape, 'MAPE_financeiro_%': base[(m, nome)],
                       'diferenca_pontos': mape - base[(m, nome)], 'DM': dm, 'p': p})
efeito = pd.DataFrame(linhas)
efeito['p_ajustado'] = avaliacao.benjamini_hochberg(efeito['p'])
efeito['leitura'] = np.select([(efeito.p_ajustado < .05) & (efeito.DM < 0), (efeito.p_ajustado < .05) & (efeito.DM > 0)],
                              ['ajuda', 'atrapalha'], 'sem efeito')
efeito.round(4).to_csv(SAIDA / 'efeito_no_modelo.csv', index=False)
print(efeito['leitura'].value_counts())
print('Sem ajuste (p < 0,05):', ((efeito.p < .05) & (efeito.DM < 0)).sum(), 'ajudam e', ((efeito.p < .05) & (efeito.DM > 0)).sum(), 'atrapalham')
efeito.sort_values('p').head(10).round(3)

## 4. Veredito por segmento

In [ ]:
veredito = efeito.pivot_table(index='segmento', columns='modelo', values='diferenca_pontos', aggfunc='mean').add_prefix('dif_media_')
veredito['ajuda'] = efeito[efeito.leitura == 'ajuda'].groupby('segmento').size()
veredito['atrapalha'] = efeito[efeito.leitura == 'atrapalha'].groupby('segmento').size()
veredito['testes_modelo'] = efeito.groupby('segmento').size()
veredito['rho_amanha_medio'] = sinal.groupby('segmento')['rho_amanha'].mean()
veredito['acerto_medio_%'] = sinal.groupby('segmento')['acerto_%'].mean()
veredito['aponta_errado'] = sinal[sinal.leitura == 'aponta o lado errado'].groupby('segmento').size()
veredito = veredito.fillna({'ajuda': 0, 'atrapalha': 0, 'aponta_errado': 0}).reindex(list(mascaras)).dropna(subset=['testes_modelo'])
veredito.round(3).to_csv(SAIDA / 'veredito.csv')
veredito.round(3)

In [ ]:
fig, eixos = plt.subplots(1, len(MODELOS), figsize=(11, 5.5), sharey=True)
lim = efeito['diferenca_pontos'].abs().max()
mapa = plt.get_cmap('RdBu_r').copy(); mapa.set_bad('#d9d9d9')
for ax, nome in zip(eixos, MODELOS):
    t = efeito[efeito.modelo == nome].pivot(index='segmento', columns='moeda', values='diferenca_pontos').reindex(index=veredito.index, columns=MOEDAS)
    im = ax.imshow(np.ma.masked_invalid(t.values), cmap=mapa, vmin=-lim, vmax=lim, aspect='auto')
    sig = efeito[(efeito.modelo == nome) & (efeito.leitura != 'sem efeito')]
    for _, l in sig.iterrows():
        ax.text(MOEDAS.index(l.moeda), list(veredito.index).index(l.segmento), '●', ha='center', va='center', fontsize=8)
    ax.set_xticks(range(len(MOEDAS))); ax.set_xticklabels(MOEDAS); ax.set_title(nome, loc='left'); ax.grid(False)
    ax.set_yticks(range(len(veredito))); ax.set_yticklabels(veredito.index)
fig.colorbar(im, ax=eixos, label='MAPE do segmento − MAPE financeiro (pontos %)\nazul = ajuda, vermelho = atrapalha')
fig.suptitle('Efeito de cada segmento de sentimento (● = significativo após ajuste; cinza = poucos dados)', x=.01, ha='left', fontsize=10)
plt.savefig(SAIDA / 'efeito_por_segmento.png', dpi=130, bbox_inches='tight'); plt.show()

In [ ]:
(SAIDA / 'config.json').write_text(json.dumps({'alvo': ALVO, 'inicio': INICIO, 'fim': FIM, 'moedas': MOEDAS,
    'sementes': config.SEMENTES, 'segmentos': list(mascaras), 'texto_forte': segmentos.FORTE,
    'min_dias_sinal': MIN_DIAS, 'min_dias_modelo': MIN_DIAS_MODELO, 'hp': MODELOS}, indent=2))
print('Pronto:', shutil.make_archive(str(SAIDA), 'zip', SAIDA))